# Aprendiendo FastAPI paso a paso

En este Notebook construiremos progresivamente una API con FastAPI.

Cada sección incorporará un concepto nuevo y conservará el código necesario para probarlo.

## 1. Instalación de FastAPI

FastAPI es el framework que utilizaremos para crear la API.

También instalaremos sus dependencias estándar, entre ellas Uvicorn, que permite ejecutar la aplicación como un servidor.

%pip install "fastapi[standard]"

## 2. Creación de la aplicación

Primero importamos la clase `FastAPI`.

Después creamos una instancia y la guardamos en la variable `app`. Este objeto representará nuestra aplicación.

In [1]:
from fastapi import FastAPI

app = FastAPI()

### Comprobación

Utilizamos 'type()' para comprobar que clase de objeto se guardo en la vaiable 'app'. 

In [2]:
type(app)

fastapi.applications.FastAPI

## 3. Ruta de prueba: Hello World

Esta ruta responde cuando alguien solicita la direccion principal de la API: '/'.

In [3]:
@app.get('/')
async def read_root():
    return{"message": "Hello, World"}

### Nota: rutas repetidas

FastAPI evalúa las rutas en el orden en que fueron registradas. Cuando recibe GET/, encuentra la primera coincidencia (read_root) y la ejecuta. Como ya encontró una ruta válida, no sigue buscando otra. Esto permite que rutas especificas se declaren antes que rutas variables, por ejemplo, /user/me antes de /user/{user_id}.

In [4]:
# Aplicamos para la misma ruta otra funcion a ver que pasa.
""" @app.get('/')
async def read_root2():
    return{"message": "Hello, World2"} """

' @app.get(\'/\')\nasync def read_root2():\n    return{"message": "Hello, World2"} '

Si dos funciones usan la misma combinación de método y ruta, por ejemplo `GET /`, FastAPI ejecuta la primera que fue registrada.

![Resultado de Hello World](Images/03%20Hello%20world.png)

Sin embargo, la documentación automática (`/docs`) solo puede mostrar una definición para `GET /`, por lo que termina mostrando la última.

Esto genera una inconsistencia: la documentación puede describir una ruta distinta de la que realmente se ejecuta. Por eso cada combinación de método HTTP y ruta debe ser única.

![Resultado de docs](Images/03%20Hello%20world%20docs.png)

## 4. Una segunda ruta: `/saludo`
Una ruta identifica una direccion de la API. `@app.get("/saludo")` registra la funcion siguiente para responder solicitudes `GET` a esa direccion.
El nombre `read_greeting` es elegido por nosotros; FastAPI usa la combinacion `GET /saludo` para encontrarla.

In [5]:
@app.get("/saludo")
async def read_greeting():
    return{"message": "Hola, Ramiro"}

![Hola Ramiro](Images\04%20Hola%20Ramiro.png)
![Hola Ramiro docs](Images\04%20Hola%20Ramiro%20docs.png)


## 5. Parametros de ruta

`{task_id}` representa una parte variable de la URL. En `/tasks/5`, FastAPI recibe 5 y lo entrega a `task_id`.
La anotacion `: init` egige un numero entero; si se escribe `/tasks/hola`, FastAPI devuelve un Error de validacion.

In [6]:
@app.get("/tasks/{task_id}")
async def read_task(task_id: int):
    return{"task_id": task_id}

![Tasks](Images\05%20Parametros%20ruta.png)
![Tasks docs](Images\05%20Parametros%20ruta%20docs.png)

## 6. Parámetros de consulta

Los parámetros de consulta son opcionales y modifican una consulta, por ejemplo `/tasks?completed=true&limit=5`; `?` inicia esa parte de la URL.

`bool | None = None` permite `True`, `False` o ausencia de filtro; así `None` significa “traer todas”.

Usamos `{task_id}` cuando el dato identifica obligatoriamente un recurso, como `/tasks/5`; usamos `?` para filtros, orden, paginación o límites que no cambian cuál es el recurso principal.

In [7]:
@app.get("/tasks")
async def read_tasks(completed: bool | None = None, limit: int = 10):
    return{"completed": completed, "limit": limit}

![Parametros consulta](Images\06%20Parametros%20consulta.png)
![Parametros consulta docs](Images\06%20Parametros%20consulta%20docs.png)

## 7. Validacion de parametros de consulta

`Query()` agrega reglas especificas para un parametro recibido desde la URL. `ge=1` significa "mayor o igual que 1" y `le=100`, "menor o igual que 100".

`Annotated` une el tipo `ìnt` con esas reglas; si `limit` queda fuera de ese rango. FastAPI rechaza la solicitud automaticamente con un error de validacion.

In [8]:
from typing import Annotated
from fastapi import Query

@app.get("/tasks-filtered")
async def read_filtered_tasks(completed:bool | None = None, limit: Annotated[int, Query(ge=1, le=100)]=10):
    return{"completed": completed, "limit": limit}

### Alcance de `Annotated` y `Query`
`Annotated` permite asociar información adicional a un tipo: aquí indica que `limit` es un `int` y que sus reglas provienen de `Query`.
Además de `ge` y `le`, `Query` puede validar longitudes, patrones, alias, valores obligatorios y descripciones para `/docs`.
El mismo mecanismo se usa más adelante con `Path`, `Header`, `Body` y `Depends`; no los aplicamos aún para incorporar una idea por vez.

![Validacion parametros consulta](Images\07%20Validacion%20parametros%20consulta.png)
![Validacion parametros consulta docs](Images\07%20Validacion%20parametros%20consulta%20docs.png)

In [ ]:
import threading, uvicorn
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000), daemon=True).start()

INFO:     Started server process [20000]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:58979 - "GET /tasks-filtered?completed=true&limit=5 HTTP/1.1" 200 OK
INFO:     127.0.0.1:58986 - "GET /tasks-filtered?completed=true&limit=105 HTTP/1.1" 422 Unprocessable Entity
INFO:     127.0.0.1:59024 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:59024 - "GET /openapi.json HTTP/1.1" 200 OK
